# Visualization of Semantic Drift

This notebook visualizes the difference between the base human-visible image and the preprocessed model-visible image using a heatmap.

In [ ]:
import sys
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Ensure repo root is in python path
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

base_path = '../data/base_images/demo_base.png'
adv_path = '../data/adversarial_images/demo_adv.png'
proc_path = '../data/processed_images/demo_proc.png'


### Load the Images

In [ ]:
# Load images using OpenCV
if os.path.exists(proc_path) and os.path.exists(adv_path):
    # We want to compare what the human sees (the adv_img rendered on a white background like a browser)
    # with what the model sees (the flattened proc_img)
    
    # Human view (simulate alpha blending on white background)
    adv_pil = Image.open(adv_path).convert("RGBA")
    white_bg = Image.new("RGB", adv_pil.size, (255, 255, 255))
    white_bg.paste(adv_pil, mask=adv_pil)
    human_view = np.array(white_bg)
    
    # Model view
    model_view = np.array(Image.open(proc_path).convert("RGB"))
else:
    print("Please run demo_attack.ipynb first to generate the images.")

### Compute Difference Heatmap

In [ ]:
if 'human_view' in locals():
    # Compute absolute difference
    diff = cv2.absdiff(human_view, model_view)
    
    # Convert difference to grayscale
    diff_gray = cv2.cvtColor(diff, cv2.COLOR_RGB2GRAY)
    
    # Create a heatmap
    heatmap = cv2.applyColorMap(diff_gray, cv2.COLORMAP_JET)
    
    # Plotting
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    
    ax[0].imshow(human_view)
    ax[0].set_title("Human View")
    ax[0].axis('off')
    
    ax[1].imshow(model_view)
    ax[1].set_title("Model View")
    ax[1].axis('off')
    
    # OpenCV uses BGR, convert heatmap to RGB for matplotlib
    ax[2].imshow(cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB))
    ax[2].set_title("Difference Heatmap")
    ax[2].axis('off')
    
    os.makedirs('../examples', exist_ok=True)
    plt.savefig('../examples/difference_heatmap.png')
    
    Image.fromarray(human_view).save('../examples/human_view.png')
    Image.fromarray(model_view).save('../examples/model_view.png')
    
    plt.show()